In [0]:
import requests
from datetime import datetime, timedelta

In [0]:
# Telegram credentials
BOT_TOKEN  = "8737861706:AAFySjSnThgiINI60fcwOxPAOZ0ULNJNCjI"
CHANNEL_ID = "-1003996075757"

# Only send jobs posted in the last 24 hours
HOURS_AGO = 24

In [0]:
# Read only jobs that haven't been sent before
new_jobs = spark.sql("""
    SELECT t.title, t.company, t.location, t.country, 
           t.job_type, t.date_posted, t.job_url, t.source
    FROM tech_jobs t
    LEFT JOIN sent_jobs s ON t.job_url = s.job_url
    WHERE s.job_url IS NULL
    ORDER BY t.date_posted DESC
    LIMIT 100
""").toPandas()

print(f"New unsent jobs : {len(new_jobs):,}")
print(new_jobs["source"].value_counts() if not new_jobs.empty else "No new jobs")

New unsent jobs : 100
source
Indeed      56
LinkedIn    44
Name: count, dtype: int64


In [0]:
def format_job(row):
    """Format a single job row into a Telegram message."""

    location = row["location"] if row["location"] not in ["NaN", "nan", "None", "Not Specified"] else "—"
    job_type  = row["job_type"]  if row["job_type"]  not in ["NaN", "nan", "None"]               else "—"
    date      = row["date_posted"] if row["date_posted"] not in ["NaN", "nan", "None", "Unknown"] else "—"

    message = (
        f"💼 {row['title']}\n"
        f"🏢 {row['company']}\n"
        f"🌍 {row['country']}  |  📍 {location}\n"
        f"⏰ {job_type}\n"
        f"📅 {date}\n"
        f"🔗 {row['job_url']}\n"
        f"📌 via {row['source']}"
    )
    return message

In [0]:
def send_message(text):
    """Send a message to the Telegram channel."""
    url  = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
    data = {
        "chat_id":    CHANNEL_ID,
        "text":       text,
        "parse_mode": "Markdown",
        "disable_web_page_preview": True,
    }
    response = requests.post(url, data=data)
    return response.ok

In [0]:
import time

if new_jobs.empty:
    print("No new jobs to send.")
else:
    success   = 0
    failed    = 0
    sent_urls = []

    for _, row in new_jobs.iterrows():
        message = format_job(row)
        ok      = send_message(message)

        if ok:
            success += 1
            sent_urls.append(row["job_url"])
        else:
            failed += 1

        # Wait 3 seconds between each message to avoid rate limiting
        time.sleep(3)

    # Save sent jobs to Delta Table
    if sent_urls:
        from pyspark.sql import Row
        from datetime import datetime

        sent_rows = [Row(job_url=url, sent_at=datetime.now()) for url in sent_urls]
        spark.createDataFrame(sent_rows).write \
            .format("delta") \
            .mode("append") \
            .saveAsTable("sent_jobs")

    print(f"✅ Sent    : {success:,}")
    print(f"❌ Failed  : {failed:,}")
    print(f"📨 Total   : {len(new_jobs):,}")

✅ Sent    : 100
❌ Failed  : 0
📨 Total   : 100


In [0]:
display(spark.sql("""
    SELECT source, COUNT(*) as total
    FROM tech_jobs
    LEFT JOIN sent_jobs ON tech_jobs.job_url = sent_jobs.job_url
    WHERE sent_jobs.job_url IS NULL
    GROUP BY source
"""))

source,total
LinkedIn,1352
Indeed,1363
